# Codelist_Cross_References — declared semantic relations between CDISC codelists

Produces `sdtm-test-codes/interim/Codelist_Cross_References.xlsx` with three sheets:

| Sheet | Grain | Source |
|---|---|---|
| `ReadMe` | — | provenance and column dictionary |
| `Cross_Reference_Pairs` | one row per declared pair | track-authored, in this notebook |
| `Term_Diff` | one row per (pair, term) in either codelist of the pair | mechanical from Cross_Reference_Pairs + SDTM_Terminology.txt |

## Why

Some CDISC codelists are semantically related but governed independently. Modern imaging sub-modes (DXA, ultrasound, MRE, DWI, MRCP, mammography) have governed METHOD terms but are absent from PROCEDUR — this gap blocks PR-side DSS authoring at right grain. The relationship between the codelists is editorial; the gap is mechanical given the relationship.

This artefact holds the small editorial declaration (which codelist pairs are semantically related, with rationale) and the mechanical derivation (per-term `in_A` / `in_B` / `gap_classification`).

## Scope discipline

The `Cross_Reference_Pairs` sheet is **track-authored**. The `Term_Diff` sheet is mechanical from declared pairs plus NCI EVS SDTM_Terminology source. New pairs land by editing the `DECLARED_PAIRS` block in this notebook and re-running.

## Inputs

| File | Track | Purpose |
|---|---|---|
| `sdtm-test-codes/downloads/SDTM_Terminology.txt` | sdtm-test-codes | NCI EVS SDTM CT package — codelists and their terms |

## Output

`sdtm-test-codes/interim/Codelist_Cross_References.xlsx`

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

In [2]:
BASE_DIR = Path.cwd().parent          # sdtm-test-codes/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

SDTM_CT_FILE = BASE_DIR / 'downloads' / 'SDTM_Terminology.txt'

INTERIM_DIR = BASE_DIR / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = INTERIM_DIR / 'Codelist_Cross_References.xlsx'

if not SDTM_CT_FILE.exists():
    raise FileNotFoundError(f'SDTM Terminology file not found: {SDTM_CT_FILE}')

print(f'  Source: {SDTM_CT_FILE.relative_to(REPO_ROOT)}')
print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')

  Source: sdtm-test-codes/downloads/SDTM_Terminology.txt
  Output: sdtm-test-codes/interim/Codelist_Cross_References.xlsx


## 2. Load SDTM Terminology

`SDTM_Terminology.txt` is tab-separated. Rows where `Codelist Code` is empty define a codelist; rows with `Codelist Code` set are governed terms within that codelist.

In [3]:
ct_raw = pd.read_csv(SDTM_CT_FILE, sep='\t', dtype=str).fillna('')
ct_raw.columns = [c.strip() for c in ct_raw.columns]
print(f'SDTM Terminology rows: {len(ct_raw):,}')
print(f'Columns: {list(ct_raw.columns)}')

# Codelist headers (rows where Codelist Code is empty)
codelists = ct_raw[ct_raw['Codelist Code'] == ''].rename(columns={
    'Code': 'codelist_concept_id',
    'Codelist Extensible (Yes/No)': 'codelist_extensible',
    'Codelist Name': 'codelist_name',
    'CDISC Submission Value': 'codelist_submission_value',
    'NCI Preferred Term': 'codelist_nci_preferred_term',
})[['codelist_concept_id', 'codelist_submission_value', 'codelist_name', 'codelist_extensible', 'codelist_nci_preferred_term']]
print(f'Codelists:        {len(codelists):,}')

# Codelist terms (rows where Codelist Code is set)
terms = ct_raw[ct_raw['Codelist Code'] != ''].rename(columns={
    'Code': 'term_concept_id',
    'Codelist Code': 'codelist_concept_id',
    'CDISC Submission Value': 'term_submission_value',
    'CDISC Synonym(s)': 'term_synonyms',
    'CDISC Definition': 'term_definition',
    'NCI Preferred Term': 'term_nci_preferred_term',
})[['term_concept_id', 'codelist_concept_id', 'term_submission_value', 'term_synonyms', 'term_definition', 'term_nci_preferred_term']]
print(f'Codelist terms:   {len(terms):,}')

SDTM Terminology rows: 46,774
Columns: ['Code', 'Codelist Code', 'Codelist Extensible (Yes/No)', 'Codelist Name', 'CDISC Submission Value', 'CDISC Synonym(s)', 'CDISC Definition', 'NCI Preferred Term']
Codelists:        1,208
Codelist terms:   45,566


## 3. Declared cross-reference pairs

Authored, not derived. Adding a pair = appending a record here with rationale.

In [4]:
DECLARED_PAIRS = [
    {
        'pair_id': 'PROCEDUR_METHOD',
        'codelist_A_concept_id': 'C101858',  # PROCEDUR
        'codelist_B_concept_id': 'C85492',   # METHOD
        'rationale': (
            'Modern imaging sub-modes (DXA, ultrasound, MRE, DWI, MRCP, '
            'mammography) have governed METHOD terms but are absent from '
            'PROCEDUR. PR-side DSS authoring at right grain is blocked '
            'behind this codelist gap. Surfaced by the consumer-side '
            'Procedure-Options Inventory (May 2026 upstream-improvements '
            'thread).'
        ),
    },
]

# Materialise as DataFrame and join codelist identity for both sides
pairs = pd.DataFrame(DECLARED_PAIRS)

cl_lookup = codelists.set_index('codelist_concept_id')
for side in ('A', 'B'):
    cid_col = f'codelist_{side}_concept_id'
    pairs[f'codelist_{side}_submission_value'] = pairs[cid_col].map(cl_lookup['codelist_submission_value'])
    pairs[f'codelist_{side}_name'] = pairs[cid_col].map(cl_lookup['codelist_name'])

# Final pair-sheet column order
pairs_out = pairs[[
    'pair_id',
    'codelist_A_concept_id', 'codelist_A_submission_value', 'codelist_A_name',
    'codelist_B_concept_id', 'codelist_B_submission_value', 'codelist_B_name',
    'rationale',
]]
print(f'Declared pairs: {len(pairs_out)}')
print(pairs_out[['pair_id', 'codelist_A_submission_value', 'codelist_B_submission_value']].to_string(index=False))

Declared pairs: 1
        pair_id codelist_A_submission_value codelist_B_submission_value
PROCEDUR_METHOD                    PROCEDUR                      METHOD


## 4. Build Term_Diff

For each declared pair, every term in (A ∪ B) gets one row with `in_codelist_A` / `in_codelist_B` booleans and a `gap_classification` (`in_both` / `in_A_only` / `in_B_only`).

In [5]:
diff_rows = []
for _, p in pairs.iterrows():
    a_terms = terms[terms['codelist_concept_id'] == p['codelist_A_concept_id']]
    b_terms = terms[terms['codelist_concept_id'] == p['codelist_B_concept_id']]
    a_set = set(a_terms['term_concept_id'])
    b_set = set(b_terms['term_concept_id'])

    # Identity lookup — prefer the row from whichever codelist holds it; concept identity is the same
    union_terms = pd.concat([a_terms, b_terms]).drop_duplicates(subset='term_concept_id')

    for _, t in union_terms.iterrows():
        tid = t['term_concept_id']
        in_a = tid in a_set
        in_b = tid in b_set
        if in_a and in_b:
            classification = 'in_both'
        elif in_a:
            classification = 'in_A_only'
        else:
            classification = 'in_B_only'
        diff_rows.append({
            'pair_id': p['pair_id'],
            'term_concept_id': tid,
            'term_submission_value': t['term_submission_value'],
            'term_nci_preferred_term': t['term_nci_preferred_term'],
            'term_definition': t['term_definition'],
            'in_codelist_A': in_a,
            'in_codelist_B': in_b,
            'gap_classification': classification,
        })

diff_out = pd.DataFrame(diff_rows).sort_values(
    ['pair_id', 'gap_classification', 'term_submission_value']
).reset_index(drop=True)

print(f'Term_Diff: {len(diff_out):,} rows x {len(diff_out.columns)} cols')
print()
print('Per-pair classification distribution:')
for pid, g in diff_out.groupby('pair_id'):
    print(f'  {pid}:')
    for cls, n in g['gap_classification'].value_counts().items():
        print(f'    {cls:12s} {n:>5}')

Term_Diff: 673 rows x 8 cols

Per-pair classification distribution:
  PROCEDUR_METHOD:
    in_B_only      521
    in_A_only      142
    in_both         10


## 5. Write workbook

Three sheets: `ReadMe`, `Cross_Reference_Pairs`, `Term_Diff`. Color convention follows the repo standard: green for SDTM-CT-side identity, grey for keys, classifications, and aggregations.

In [6]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER = PatternFill('solid', fgColor='548235')   # TESTCD / SDTM CT side
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # COSMoS side
GREY_HEADER = PatternFill('solid', fgColor='808080')    # keys, aggregation


def write_sheet(ws, df, header_fills, col_widths):
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            if isinstance(val, bool):
                val = 'Y' if val else 'N'
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'


print('Writer ready.')

Writer ready.


In [7]:
wb = Workbook()
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('Codelist_Cross_References — declared semantic relations between CDISC codelists', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Notebook: sdtm-test-codes/notebooks/Codelist_Cross_References.ipynb', readme_font),
    (f'Inputs:', readme_font),
    (f'  sdtm-test-codes/downloads/SDTM_Terminology.txt (NCI EVS SDTM CT)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('Editorial declaration of which CDISC codelist pairs are semantically', readme_font),
    ('related, plus mechanical derivation of the per-term overlap and gap.', readme_font),
    ('', None),
    ('AUTHORING DISCIPLINE', section_font),
    ('Cross_Reference_Pairs is track-authored. New pairs land by editing', readme_font),
    ('the DECLARED_PAIRS block in the notebook and re-running. Term_Diff', readme_font),
    ('is mechanical given declared pairs + SDTM_Terminology source.', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Cross_Reference_Pairs — one row per declared semantically-related pair.', readme_font),
    ('  pair_id                       — short identifier', readme_font),
    ('  codelist_A_concept_id         — NCIt C-code, side A', readme_font),
    ('  codelist_A_submission_value   — CDISC submission value, side A', readme_font),
    ('  codelist_A_name               — CDISC codelist name, side A', readme_font),
    ('  codelist_B_*                  — same, side B', readme_font),
    ('  rationale                     — why these are declared related', readme_font),
    ('Term_Diff — one row per (pair, term) in (A union B).', readme_font),
    ('  pair_id                       — FK to Cross_Reference_Pairs', readme_font),
    ('  term_concept_id               — NCIt C-code', readme_font),
    ('  term_submission_value         — CDISC submission value', readme_font),
    ('  term_nci_preferred_term       — NCI preferred term', readme_font),
    ('  term_definition               — CDISC definition', readme_font),
    ('  in_codelist_A                 — Y/N: term is governed in side A', readme_font),
    ('  in_codelist_B                 — Y/N: term is governed in side B', readme_font),
    ('  gap_classification            — in_both / in_A_only / in_B_only', readme_font),
    ('', None),
    ('SEED DECLARATION (2026-05)', section_font),
    ('PROCEDUR (C101858) <-> METHOD (C85492). Motivated by the May 2026', readme_font),
    ('upstream-improvements thread: imaging sub-modes governed in METHOD', readme_font),
    ('but absent from PROCEDUR block PR-side DSS authoring at right grain.', readme_font),
    ('Other candidate pairs from the proposal (LOC <-> DIR, LBSPEC /', readme_font),
    ('MSSPEC / MISPEC) land as the rationale is authored.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('First declared cross-reference at the sdtm-test-codes layer.', readme_font),
    ('Sources: NCI EVS SDTM CT package 2026-03-27.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f'ReadMe: {len(readme_lines)} lines')

ReadMe: 45 lines


In [8]:
# ── Cross_Reference_Pairs sheet ──
ws_p = wb.create_sheet('Cross_Reference_Pairs')

P_FILLS = {
    'pair_id':                       GREY_HEADER,
    'codelist_A_concept_id':         GREY_HEADER,
    'codelist_A_submission_value':   GREEN_HEADER,
    'codelist_A_name':               GREEN_HEADER,
    'codelist_B_concept_id':         GREY_HEADER,
    'codelist_B_submission_value':   GREEN_HEADER,
    'codelist_B_name':               GREEN_HEADER,
    'rationale':                     GREY_HEADER,
}

P_WIDTHS = {
    'pair_id':                       18,
    'codelist_A_concept_id':         14,
    'codelist_A_submission_value':   18,
    'codelist_A_name':               30,
    'codelist_B_concept_id':         14,
    'codelist_B_submission_value':   18,
    'codelist_B_name':               30,
    'rationale':                     80,
}

write_sheet(ws_p, pairs_out, P_FILLS, P_WIDTHS)
print(f'Cross_Reference_Pairs: {len(pairs_out)} rows x {len(pairs_out.columns)} cols')

Cross_Reference_Pairs: 1 rows x 8 cols


In [9]:
# ── Term_Diff sheet ──
ws_t = wb.create_sheet('Term_Diff')

T_FILLS = {
    'pair_id':                  GREY_HEADER,
    'term_concept_id':          GREY_HEADER,
    'term_submission_value':    GREEN_HEADER,
    'term_nci_preferred_term':  GREEN_HEADER,
    'term_definition':          GREEN_HEADER,
    'in_codelist_A':            GREY_HEADER,
    'in_codelist_B':            GREY_HEADER,
    'gap_classification':       GREY_HEADER,
}

T_WIDTHS = {
    'pair_id':                  18,
    'term_concept_id':          14,
    'term_submission_value':    32,
    'term_nci_preferred_term':  35,
    'term_definition':          60,
    'in_codelist_A':            12,
    'in_codelist_B':            12,
    'gap_classification':       16,
}

write_sheet(ws_t, diff_out, T_FILLS, T_WIDTHS)
print(f'Term_Diff: {len(diff_out):,} rows x {len(diff_out.columns)} cols')

Term_Diff: 673 rows x 8 cols


In [10]:
wb.save(OUTPUT_FILE)
print(f'\nWritten: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')


Written: /sessions/zealous-wonderful-einstein/mnt/cdisc-for-ai/sdtm-test-codes/interim/Codelist_Cross_References.xlsx
File size: 88 KB


## 6. Summary

In [11]:
print('=== Codelist_Cross_References summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Cross_Reference_Pairs: {len(pairs_out)} rows x {len(pairs_out.columns)} cols')
print(f'Term_Diff:             {len(diff_out):,} rows x {len(diff_out.columns)} cols')
print()
for _, p in pairs_out.iterrows():
    g = diff_out[diff_out['pair_id'] == p['pair_id']]
    a_only = (g['gap_classification'] == 'in_A_only').sum()
    b_only = (g['gap_classification'] == 'in_B_only').sum()
    both   = (g['gap_classification'] == 'in_both').sum()
    print(f'{p["pair_id"]} ({p["codelist_A_submission_value"]} <-> {p["codelist_B_submission_value"]}):')
    print(f'  in {p["codelist_A_submission_value"]} only: {a_only}')
    print(f'  in {p["codelist_B_submission_value"]} only: {b_only}')
    print(f'  in both:               {both}')
    print(f'  union:                 {len(g)}')
print()
# Spot-check the inventory's named imaging sub-modes — should appear as in_B_only.
# Submission values used here match CDISC governance (the inventory's clinical
# abbreviations MRE/DWI/MRCP map to fuller submission values in METHOD).
INVENTORY_NAMED = [
    'DXA SCAN',
    'ULTRASOUND',
    'MAMMOGRAPHY',
    'MAGNETIC RESONANCE ELASTOGRAPHY',
    'DIFFUSION WEIGHTED MRI',
    'MAGNETIC RESONANCE CHOLANGIOPANCREATOGRAPHY',
]
print('Inventory-named imaging sub-modes (expected: in METHOD only, gap from PROCEDUR):')
for term in INVENTORY_NAMED:
    hit = diff_out[diff_out['term_submission_value'] == term]
    if hit.empty:
        print(f'  {term:50s}  not found in either codelist')
    else:
        for _, r in hit.iterrows():
            print(f'  {term:50s}  pair={r["pair_id"]:20s} {r["gap_classification"]}')


=== Codelist_Cross_References summary ===
Output: sdtm-test-codes/interim/Codelist_Cross_References.xlsx

Cross_Reference_Pairs: 1 rows x 8 cols
Term_Diff:             673 rows x 8 cols

PROCEDUR_METHOD (PROCEDUR <-> METHOD):
  in PROCEDUR only: 142
  in METHOD only: 521
  in both:               10
  union:                 673

Inventory-named imaging sub-modes (expected: in METHOD only, gap from PROCEDUR):
  DXA SCAN                                            pair=PROCEDUR_METHOD      in_B_only
  ULTRASOUND                                          pair=PROCEDUR_METHOD      in_B_only
  MAMMOGRAPHY                                         pair=PROCEDUR_METHOD      in_B_only
  MAGNETIC RESONANCE ELASTOGRAPHY                     pair=PROCEDUR_METHOD      in_B_only
  DIFFUSION WEIGHTED MRI                              pair=PROCEDUR_METHOD      in_B_only
  MAGNETIC RESONANCE CHOLANGIOPANCREATOGRAPHY         pair=PROCEDUR_METHOD      in_B_only
